In [1]:
import pandas as pd
df = pd.read_csv('../data/raw/all_votes_person_id.csv')

x = (df['ja/nein'] == df['janein']).all()
print(x)

True


In [4]:
import json

chemin = '../data/processed/bundestag_signed.json' 

with open(chemin, 'r') as f:
    graph_data = json.load(f)

print(graph_data.keys())
print(graph_data['edges'][0])

dict_keys(['edges'])
{'source': 0, 'target': 1, 'weight': 1}


In [3]:
import pandas as pd

print("Chargement des données...")
df = pd.read_csv("../data/raw/all_votes_person_id.csv")

# Nettoyage des noms de partis pour avoir des groupes propres
df['Fraktion/Gruppe'] = df['Fraktion/Gruppe'].replace({
    'DIE LINKE.': 'DIE LINKE',
    'BÜNDNIS`90/DIE GRÜNEN': 'GRÜNE',
    'BÜ90/GR': 'GRÜNE'
})

# --- LES BONNES COLONNES ---
COLONNE_SCRUTIN = 'filename' 
COLONNE_VOTE = 'ja/nein'

print("Calcul des majorités par parti et par vote...")
# On cherche le vote majoritaire de chaque parti pour chaque loi (filename)
majority_votes = df.groupby([COLONNE_SCRUTIN, 'Fraktion/Gruppe'])[COLONNE_VOTE].agg(lambda x: x.mode()[0]).reset_index()

# On crée un tableau croisé : Scrutins en lignes, Partis en colonnes
pivot_df = majority_votes.pivot(index=COLONNE_SCRUTIN, columns='Fraktion/Gruppe', values=COLONNE_VOTE)

# On garde uniquement les scrutins où ces trois partis étaient présents
pivot_df = pivot_df.dropna(subset=['AfD', 'DIE LINKE', 'CDU/CSU'])

# --- LES STATISTIQUES ---
total_votes = len(pivot_df)

linke_afd_agree = (pivot_df['DIE LINKE'] == pivot_df['AfD']).sum()
afd_cdu_agree = (pivot_df['AfD'] == pivot_df['CDU/CSU']).sum()

print("\n=== RÉSULTATS DE LA VÉRIFICATION ===")
print(f"Nombre total de scrutins analysés : {total_votes}")
print(f"L'AfD et Die Linke ont voté EXACTEMENT PAREIL sur {linke_afd_agree} scrutins ({(linke_afd_agree/total_votes)*100:.1f}%)")
print(f"L'AfD et la CDU ont voté EXACTEMENT PAREIL sur {afd_cdu_agree} scrutins ({(afd_cdu_agree/total_votes)*100:.1f}%)")

# On affiche un extrait pour voir de quoi il s'agit
print("\nExemple de votes (ja, nein, enthalten) où Linke et AfD étaient d'accord :")
common_opposition = pivot_df[pivot_df['DIE LINKE'] == pivot_df['AfD']][['DIE LINKE', 'AfD', 'CDU/CSU']].head(10)
print(common_opposition)

Chargement des données...
Calcul des majorités par parti et par vote...

=== RÉSULTATS DE LA VÉRIFICATION ===
Nombre total de scrutins analysés : 334
L'AfD et Die Linke ont voté EXACTEMENT PAREIL sur 164 scrutins (49.1%)
L'AfD et la CDU ont voté EXACTEMENT PAREIL sur 127 scrutins (38.0%)

Exemple de votes (ja, nein, enthalten) où Linke et AfD étaient d'accord :
Fraktion/Gruppe    DIE LINKE   AfD CDU/CSU
filename                                  
20171212_2_xls.xls      nein  nein      ja
20171212_3_xls.xls      nein  nein      ja
20171212_4_xls.xls      nein  nein      ja
20171212_5_xls.xls      nein  nein      ja
20171213_3_xls.xls      nein  nein      ja
20180201_1_xls.xls      nein  nein      ja
20180322_1_xls.xls      nein  nein      ja
20180322_2_xls.xls      nein  nein      ja
20180322_3_xls.xls      nein  nein      ja
20180426_1_xls.xls      nein  nein      ja
